# SpikeFormer Benchmark - Kaggle GPU

Benchmarks **ANN Transformer** vs **SNN SpikeFormer** on CIFAR-10.
Run `train_spikeformer.ipynb` first to train models and download checkpoints.

Reference: [Xpikeformer paper (arXiv:2408.08794v2)](https://arxiv.org/abs/2408.08794v2)

## Setup

In [ ]:
import os
import sys
sys.path.insert(0, '.')

import torch
import numpy as np
from torch.utils.data import DataLoader
from torchvision import datasets, transforms

from src.ann.transformer import CIFAR10ANNTransformer, create_ann_transformer
from src.snn.spikeformer import CIFAR10Spikeformer, create_spikeformer
from src.snn.benchmark import Benchmark, EnergyEstimates

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

In [ ]:
# Upload checkpoints via Kaggle UI, then:
# !unzip ann_checkpoints.zip -d .
# !unzip snn_checkpoints.zip -d .

ANN_CHECKPOINT = 'checkpoints_ann/ann_best.pth'
SNN_CHECKPOINT = 'checkpoints/snn_best.pth'
TIMESTEPS = 4

print(f"ANN checkpoint: {ANN_CHECKPOINT} (exists: {os.path.exists(ANN_CHECKPOINT)})")
print(f"SNN checkpoint: {SNN_CHECKPOINT} (exists: {os.path.exists(SNN_CHECKPOINT)})")

## Load Models

In [ ]:
# Load ANN model
ann_model = create_ann_transformer().to(device)
if os.path.exists(ANN_CHECKPOINT):
    checkpoint = torch.load(ANN_CHECKPOINT, map_location=device)
    ann_model.load_state_dict(checkpoint['model_state_dict'])
    print(f"Loaded ANN checkpoint (epoch {checkpoint.get('epoch', '?')}, acc {checkpoint.get('test_acc', 0):.2f}%)")
else:
    print("No ANN checkpoint found - using untrained model")

ann_params = sum(p.numel() for p in ann_model.parameters())
print(f"ANN params: {ann_params:,}")

In [ ]:
# Load SNN model
snn_model = create_spikeformer(timesteps=TIMESTEPS).to(device)
if os.path.exists(SNN_CHECKPOINT):
    checkpoint = torch.load(SNN_CHECKPOINT, map_location=device)
    snn_model.load_state_dict(checkpoint['model_state_dict'])
    print(f"Loaded SNN checkpoint (epoch {checkpoint.get('epoch', '?')}, acc {checkpoint.get('test_acc', 0):.2f}%)")
else:
    print("No SNN checkpoint found - using untrained model")

snn_params = sum(p.numel() for p in snn_model.parameters())
print(f"SNN params: {snn_params:,}")

## Data Loading

In [ ]:
transform_test = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.4914, 0.4822, 0.4465), (0.2023, 0.1994, 0.2010))
])

test_dataset = datasets.CIFAR10(root='./data', train=False, download=True, transform=transform_test)
test_loader = DataLoader(test_dataset, batch_size=100, shuffle=False, num_workers=2)

print(f"Test samples: {len(test_dataset)}")

## Accuracy Benchmark

In [ ]:
def evaluate_accuracy(model, loader, device, is_snn=False, timesteps=4):
    """Evaluate model accuracy on test set."""
    model.eval()
    correct = 0
    total = 0
    
    with torch.no_grad():
        for data, target in loader:
            data, target = data.to(device), target.to(device)
            
            if is_snn:
                output = model(data, timesteps=timesteps)
            else:
                output = model(data)
            
            _, predicted = output.max(1)
            total += target.size(0)
            correct += predicted.eq(target).sum().item()
    
    return 100. * correct / total

print("Evaluating ANN...")
ann_acc = evaluate_accuracy(ann_model, test_loader, device, is_snn=False)
print(f"ANN Test Accuracy: {ann_acc:.2f}%")

print("\nEvaluating SNN...")
snn_acc = evaluate_accuracy(snn_model, test_loader, device, is_snn=True, timesteps=TIMESTEPS)
print(f"SNN Test Accuracy: {snn_acc:.2f}%")

## Latency Benchmark

In [ ]:
import time

def benchmark_latency(model, device, is_snn=False, timesteps=4, num_runs=100, warmup=10):
    """Benchmark inference latency."""
    model.eval()
    dummy_input = torch.randn(1, 3, 32, 32).to(device)
    
    # Warmup
    with torch.no_grad():
        for _ in range(warmup):
            if is_snn:
                _ = model(dummy_input, timesteps=timesteps)
            else:
                _ = model(dummy_input)
    
    # Benchmark
    if device.type == 'cuda':
        torch.cuda.synchronize()
    
    times = []
    with torch.no_grad():
        for _ in range(num_runs):
            if device.type == 'cuda':
                torch.cuda.synchronize()
            
            start = time.perf_counter()
            if is_snn:
                _ = model(dummy_input, timesteps=timesteps)
            else:
                _ = model(dummy_input)
            
            if device.type == 'cuda':
                torch.cuda.synchronize()
            
            times.append(time.perf_counter() - start)
    
    return np.mean(times) * 1000, np.std(times) * 1000  # ms

print("Benchmarking ANN latency...")
ann_mean, ann_std = benchmark_latency(ann_model, device, is_snn=False)
print(f"ANN: {ann_mean:.2f} ± {ann_std:.2f} ms")

print("\nBenchmarking SNN latency...")
snn_mean, snn_std = benchmark_latency(snn_model, device, is_snn=True, timesteps=TIMESTEPS)
print(f"SNN: {snn_mean:.2f} ± {snn_std:.2f} ms")

## Energy Estimation

In [ ]:
# Use the benchmark module's energy estimates
energy = EnergyEstimates()

# Estimate FLOPs (simplified)
def estimate_flops(params, latency_ms, is_snn=False, timesteps=4):
    """Rough FLOPs estimate based on parameters and latency."""
    if is_snn:
        # SNN: each param used T times for temporal integration
        return params * timesteps * 2  # multiply-add per param per timestep
    else:
        return params * 2  # multiply-add per param

ann_flops = estimate_flops(ann_params, ann_mean, is_snn=False)
snn_flops = estimate_flops(snn_params, snn_mean, is_snn=True, timesteps=TIMESTEPS)

ann_energy_pj = energy.estimate_energy(ann_flops, is_snn=False)
snn_energy_pj = energy.estimate_energy(snn_flops, is_snn=True)

print(f"ANN: ~{ann_flops:,} FLOPs, ~{ann_energy_pj:.2f} pJ/sample")
print(f"SNN: ~{snn_flops:,} FLOPs, ~{snn_energy_pj:.2f} pJ/sample")

## Summary

In [ ]:
print("\n" + "="*60)
print("BENCHMARK RESULTS SUMMARY")
print("="*60)
print(f"{'Metric':<20} {'ANN':>15} {'SNN':>15}")
print("-"*60)
print(f"{'Parameters':<20} {ann_params:>15,} {snn_params:>15,}")
print(f"{'Test Accuracy':<20} {ann_acc:>14.2f}% {snn_acc:>14.2f}%")
print(f"{'Latency (ms)':<20} {ann_mean:>15.2f} {snn_mean:>15.2f}")
print(f"{'Energy (pJ/sample)':<20} {ann_energy_pj:>15.2f} {snn_energy_pj:>15.2f}")
print("="*60)
print(f"\nSNN uses {snn_params/ann_params*100:.1f}% of ANN parameters")
print(f"SNN is {snn_mean/ann_mean:.1f}x slower on GPU (simulated)")
print(f"\nNote: On real neuromorphic hardware, SNN would be much faster")
print(f"      due to parallel temporal processing and event-driven operations.")